In [1]:
import sys
sys.path.append("..")

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import root_mean_squared_error, r2_score
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from src.data import load_glass_data
from src.model import save_model, load_model

TARGETS = ["Tg", "Density293K"]
RANDOM_STATE = 42
TEST_SIZE = 0.2

In [2]:
datasets = {}

for target in TARGETS:
    df = load_glass_data(target=target)
    features = [c for c in df.columns if c != target]
    X = df[features]
    y = df[target]
    datasets[target] = {"X": X, "y": y, "features": features}
    print(f"{target}: {X.shape[0]} rows, {X.shape[1]} features")

Tg: 49972 rows, 22 features
Density293K: 61157 rows, 21 features


In [3]:
splits = {}

for target, data in datasets.items():
    X_train, X_test, y_train, y_test = train_test_split(
        data["X"], data["y"], test_size=TEST_SIZE, random_state=RANDOM_STATE
    )
    splits[target] = {
        "X_train": X_train, "X_test": X_test,
        "y_train": y_train, "y_test": y_test,
    }
    print(f"{target}: {X_train.shape[0]} train, {X_test.shape[0]} test")

Tg: 39977 train, 9995 test
Density293K: 48925 train, 12232 test


In [4]:
baselines = {
    "RF": RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1),
    "XGBoost": XGBRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1),
    "LightGBM": LGBMRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1),
}

baseline_results = {target: {} for target in TARGETS}

for target, split in splits.items():
    print(f"\n--- {target} ---")
    for name, model in baselines.items():
        model.fit(split["X_train"], split["y_train"])
        preds = model.predict(split["X_test"])
        rmse = root_mean_squared_error(split["y_test"], preds)
        r2 = r2_score(split["y_test"], preds)
        baseline_results[target][f"Baseline {name}"] = {"RMSE": rmse, "R2": r2}
        print(f"  {name:<10} RMSE: {rmse:.4f}  R²: {r2:.4f}")


--- Tg ---
  RF         RMSE: 33.8808  R²: 0.9421
  XGBoost    RMSE: 38.3985  R²: 0.9256
  LightGBM   RMSE: 42.1621  R²: 0.9103

--- Density293K ---
  RF         RMSE: 0.2103  R²: 0.9618
  XGBoost    RMSE: 0.2140  R²: 0.9604
  LightGBM   RMSE: 0.2314  R²: 0.9537


In [5]:
PARAM_GRIDS = {
    "RF": {
        "estimator": RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
        "params": {
            "n_estimators": [200, 300, 500],
            "max_depth": [20, 30, 40, None],
            "max_features": [0.3, 0.5, 0.7],
            "min_samples_leaf": [1, 2, 4],
        }
    },
    "XGBoost": {
        "estimator": XGBRegressor(random_state=RANDOM_STATE, n_jobs=-1),
        "params": {
            "n_estimators": [200, 300, 500],
            "max_depth": [4, 6, 8],
            "learning_rate": [0.05, 0.1, 0.2],
            "subsample": [0.7, 0.8, 1.0],
            "colsample_bytree": [0.7, 0.8, 1.0],
        }
    },
    "LightGBM": {
        "estimator": LGBMRegressor(random_state=RANDOM_STATE, n_jobs=-1, verbose=-1),
        "params": {
            "n_estimators": [200, 300, 500],
            "max_depth": [4, 6, 8],
            "learning_rate": [0.05, 0.1, 0.2],
            "subsample": [0.7, 0.8, 1.0],
            "colsample_bytree": [0.7, 0.8, 1.0],
            "num_leaves": [31, 63, 127],
        }
    },
}

tuned_results = {target: {} for target in TARGETS}

for target, split in splits.items():
    print(f"\n=== {target} ===")
    for name, cfg in PARAM_GRIDS.items():
        search = RandomizedSearchCV(
            cfg["estimator"],
            param_distributions=cfg["params"],
            n_iter=20,
            cv=5,
            scoring="neg_root_mean_squared_error",
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbose=1,
        )
        search.fit(split["X_train"], split["y_train"])
        preds = search.best_estimator_.predict(split["X_test"])
        rmse = root_mean_squared_error(split["y_test"], preds)
        r2 = r2_score(split["y_test"], preds)
        tuned_results[target][f"Tuned {name}"] = {
            "model": search.best_estimator_,
            "RMSE": rmse,
            "R2": r2,
            "best_params": search.best_params_,
        }
        print(f"  {name:<10} RMSE: {rmse:.4f}  R²: {r2:.4f}  params: {search.best_params_}")


=== Tg ===
Fitting 5 folds for each of 20 candidates, totalling 100 fits
  RF         RMSE: 32.3859  R²: 0.9471  params: {'n_estimators': 500, 'min_samples_leaf': 1, 'max_features': 0.3, 'max_depth': 40}
Fitting 5 folds for each of 20 candidates, totalling 100 fits
  XGBoost    RMSE: 33.5629  R²: 0.9432  params: {'subsample': 1.0, 'n_estimators': 500, 'max_depth': 8, 'learning_rate': 0.1, 'colsample_bytree': 1.0}
Fitting 5 folds for each of 20 candidates, totalling 100 fits
  LightGBM   RMSE: 36.4942  R²: 0.9328  params: {'subsample': 0.7, 'num_leaves': 127, 'n_estimators': 500, 'max_depth': 8, 'learning_rate': 0.05, 'colsample_bytree': 0.7}

=== Density293K ===
Fitting 5 folds for each of 20 candidates, totalling 100 fits
  RF         RMSE: 0.2015  R²: 0.9649  params: {'n_estimators': 500, 'min_samples_leaf': 1, 'max_features': 0.3, 'max_depth': 40}
Fitting 5 folds for each of 20 candidates, totalling 100 fits
  XGBoost    RMSE: 0.2027  R²: 0.9644  params: {'subsample': 0.7, 'n_estim

In [6]:
for target in TARGETS:
    print(f"\n=== {target} ===")
    print(f"{'Model':<20} {'RMSE':>12} {'R²':>10}")
    print("-" * 44)
    for name, metrics in baseline_results[target].items():
        print(f"{name:<20} {metrics['RMSE']:>12.4f} {metrics['R2']:>10.4f}")
    for name, metrics in tuned_results[target].items():
        print(f"{name:<20} {metrics['RMSE']:>12.4f} {metrics['R2']:>10.4f}")


=== Tg ===
Model                        RMSE         R²
--------------------------------------------
Baseline RF               33.8808     0.9421
Baseline XGBoost          38.3985     0.9256
Baseline LightGBM         42.1621     0.9103
Tuned RF                  32.3859     0.9471
Tuned XGBoost             33.5629     0.9432
Tuned LightGBM            36.4942     0.9328

=== Density293K ===
Model                        RMSE         R²
--------------------------------------------
Baseline RF                0.2103     0.9618
Baseline XGBoost           0.2140     0.9604
Baseline LightGBM          0.2314     0.9537
Tuned RF                   0.2015     0.9649
Tuned XGBoost              0.2027     0.9644
Tuned LightGBM             0.2088     0.9623


In [7]:
for target in TARGETS:
    best_name = min(tuned_results[target], key=lambda k: tuned_results[target][k]["RMSE"])
    best = tuned_results[target][best_name]
    
    save_model(
        model=best["model"],
        path=f"../outputs/models/{target.lower()}_best.joblib",
        metadata={
            "target": target,
            "features": datasets[target]["features"],
            "model": best_name,
            "test_rmse": best["RMSE"],
            "test_r2": best["R2"],
            "best_params": best["best_params"],
        }
    )

Saved to ..\outputs\models\tg_best.joblib
Saved to ..\outputs\models\density293k_best.joblib


In [8]:
for target in TARGETS:
    model, metadata = load_model(f"../outputs/models/{target.lower()}_best.joblib")
    print(f"{target}: {metadata['model']} — RMSE {metadata['test_rmse']:.4f}, R² {metadata['test_r2']:.4f}")

Tg: Tuned RF — RMSE 32.3859, R² 0.9471
Density293K: Tuned RF — RMSE 0.2015, R² 0.9649


## Key findings

### Results summary
- Tuned RF wins on both properties
- Tg: RMSE 32.39°C, R² 0.947 — predicting within ~32°C on average
- Density293K: RMSE 0.20 g/cm³, R² 0.965 — very strong

### Tuning observations
- RF consistently picked `max_features: 0.3` and `min_samples_leaf: 1` on both 
  properties — worth hardcoding as starting point for future properties
- LightGBM underperforms both RF and XGBoost even after tuning — may need more 
  iterations (n_iter=20 is conservative) or a wider param grid
- Density models cluster tightly after tuning (0.2015–0.2088) vs Tg where there's 
  more spread (32.39–36.49) — density has more linear relationships, easier to learn

### Next steps
- Increase n_iter for a more thorough search — 20 is conservative
- Try neural networks (MLP) as an alternative to tree-based models
- Expand to other properties (Tmelt, Tsoft, etc.)
- Consider feature engineering — oxide ratios, network connectivity indices